# Basins attributes

In [ ]:
import xarray as xr
import pandas as pd
import geopandas as gpd

from andeangc import config as cfg
from andeangc import basin_attributes as ba
from andeangc.polygon_extract import extract_attributes

## AndeanGC data

In [ ]:
AndeanGC_metadata = pd.read_csv(cfg.VERSION / 'AndeanGC_metadata.csv')
AndeanGC_shape    = gpd.read_file(cfg.VERSION / 'AndeanGC_shape.gpkg')

## Geometry and topography

In [ ]:
AndeanGC_shape = ba.geometry_attributes(AndeanGC_shape)
AndeanGC_shape = ba.topographic_attributes(AndeanGC_shape)

## Climate

In [ ]:
climate_stack = ba.era5_reference_stack()

# prcp_mean
stack = climate_stack.prcp.resample(time='YS').sum(dim='time').mean(dim='time')
AndeanGC_shape = extract_attributes(stack, AndeanGC_shape, "prcp_mean_ERA5")

# prcp_pci
stack = climate_stack.prcp.resample(time='MS').sum(dim='time').groupby('time.month').mean(dim='time')
stack = (stack ** 2).sum(dim='month') * 100 / (stack.sum(dim='month') ** 2)
AndeanGC_shape = extract_attributes(stack, AndeanGC_shape, "prcp_pci_ERA5")

# ev_mean and aridity
stack = climate_stack.ep.resample(time='YS').sum(dim='time').mean(dim='time')
AndeanGC_shape = extract_attributes(stack, AndeanGC_shape, "ev_mean_ERA5")
AndeanGC_shape['aridity_ERA5'] = AndeanGC_shape['ev_mean_ERA5'] / AndeanGC_shape['prcp_mean_ERA5']

# solid prcp (< 0◦C)
stack = xr.where(climate_stack.tas > 0, 0, climate_stack.prcp)
stack = stack.resample(time='YS').sum(dim='time').mean(dim='time')
AndeanGC_shape = extract_attributes(stack, AndeanGC_shape, "solid_prcp_ERA5")
AndeanGC_shape['frac_snow_ERA5'] = AndeanGC_shape['solid_prcp_ERA5'] / AndeanGC_shape['prcp_mean_ERA5']

# prcp freq
stack = (climate_stack.prcp < 1).sum(dim='time')
stack = climate_stack.prcp.sizes['time'] / stack
AndeanGC_shape = extract_attributes(stack, AndeanGC_shape, "low_prec_freq_ERA5")

stack = (climate_stack.prcp > climate_stack.prcp.mean(dim='time') * 5).sum(dim='time')
stack = climate_stack.prcp.sizes['time'] / stack
AndeanGC_shape = extract_attributes(stack, AndeanGC_shape, "high_prec_freq_ERA5")

## Glacier, land cover and dams

In [ ]:
AndeanGC_shape = ba.glacier_attributes(AndeanGC_shape)
AndeanGC_shape = ba.land_cover_attributes(AndeanGC_shape)
AndeanGC_shape = ba.dam_attributes(AndeanGC_shape)

## Save (and overwrite) data


In [ ]:
# Drop geometry and select only new columns not in AndeanGC_metadata
AndeanGC_shape_subset = AndeanGC_shape.drop(columns='geometry')
AndeanGC_shape_subset = AndeanGC_shape_subset.loc[:, ~AndeanGC_shape_subset.columns.isin(AndeanGC_metadata.columns)]
AndeanGC_metadata = AndeanGC_metadata.join(AndeanGC_shape_subset, on='gauge_id', how='left')

# Save outputs
AndeanGC_shape.to_file(cfg.VERSION / 'AndeanGC_shape.gpkg')
AndeanGC_metadata.round(4).to_csv(cfg.VERSION / 'AndeanGC_metadata.csv', index=False)

In [ ]:
# TODO leaf area index (LAI) -> this is missing
#lai_data = rxr.open_rasterio(path_data_raw + "GIS/LAI_MOD15A2H_climatology.nc")
#lai_data = lai_data.mean(dim="time", skipna=True)
#lai_data = exact_extract(lai_data, basin_shp, "mean", progress=False)
#basin_shp["lai_max"] = lai_data.max(axis=1, skipna=True)
#basin_shp["lai_diff"] = lai_data.max(axis=1, skipna=True) - lai_data.min(axis=1, skipna=True)